# SQL Playground
DuckDB + Jupyter. Everything you create is stored in `work/playground.duckdb` and survives restarts.

1. Define a schema with SQL → draw the ER diagram
2. Or draw the ER diagram as text (Mermaid) → get the schema generated
3. Play with queries

In [ ]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
import erplay
con = erplay.connect()
%sql con

## 1. Schema with SQL
Re-running this cell is safe: it starts with a clean database.

In [ ]:
erplay.reset(con)

In [ ]:
%%sql
CREATE TABLE student (
    id    INTEGER PRIMARY KEY,
    name  VARCHAR NOT NULL,
    class VARCHAR
);
CREATE TABLE teacher (
    id   INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL
);
CREATE TABLE course (
    id         INTEGER PRIMARY KEY,
    title      VARCHAR NOT NULL,
    teacher_id INTEGER REFERENCES teacher(id)
);
CREATE TABLE enrollment (
    student_id INTEGER REFERENCES student(id),
    course_id  INTEGER REFERENCES course(id),
    grade      INTEGER CHECK (grade BETWEEN 1 AND 6),
    PRIMARY KEY (student_id, course_id)
);

In [ ]:
%%sql
INSERT INTO student VALUES (1,'Anna','10a'),(2,'Ben','10a'),(3,'Chris','10b');
INSERT INTO teacher VALUES (1,'Ms Meyer'),(2,'Mr Schulz');
INSERT INTO course VALUES (1,'Math',1),(2,'Physics',2),(3,'Art',NULL);
INSERT INTO enrollment VALUES (1,1,2),(1,2,1),(2,1,3),(3,3,2);

## 2. ER diagram of the current database

In [ ]:
erplay.er_diagram(con)

The same schema as editable Mermaid text (paste it into any Mermaid renderer, e.g. GitHub markdown or mermaid.live):

In [ ]:
print(erplay.schema_to_mermaid(con))

## 3. Queries

In [ ]:
%%sql
SELECT s.name AS student, c.title AS course, e.grade
FROM enrollment e
JOIN student s ON s.id = e.student_id
JOIN course  c ON c.id = e.course_id
ORDER BY s.name;

In [ ]:
%%sql
-- average grade per course, including courses nobody takes
SELECT c.title, round(avg(e.grade), 2) AS avg_grade, count(e.student_id) AS students
FROM course c LEFT JOIN enrollment e ON e.course_id = c.id
GROUP BY c.title
ORDER BY avg_grade;

Your turn: write your own queries below.

In [ ]:
%%sql
-- your query here
SELECT * FROM student;

## 4. ER diagram → schema
Describe entities and relationships as a Mermaid `erDiagram`; the tables and foreign keys are generated.

* `A ||--o{ B` = one A has zero or many B (B gets a foreign key to A)
* `|o` / `o|` = the parent is optional (nullable foreign key)
* `}o--o{` = many-to-many (a junction table is created)
* Mark keys with `PK`, `FK`, `UK` after the column name

This wipes the database first.

In [ ]:
erplay.reset(con)
diagram = """
erDiagram
    AUTHOR ||--o{ BOOK : writes
    MEMBER ||--o{ LOAN : makes
    BOOK   ||--o{ LOAN : "is lent in"
    AUTHOR {
        int id PK
        string name
    }
    BOOK {
        int id PK
        string title
        int year
    }
    MEMBER {
        int id PK
        string name
        string email UK
    }
    LOAN {
        int id PK
        date loaned_on
    }
"""
erplay.apply_mermaid(con, diagram)